In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
train = pd.read_csv("/content/train.csv")
sample = pd.read_csv("/content/sample_submission.csv")

In [92]:
train['date'] = pd.to_datetime(train['date'], errors='coerce')
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['week'] = train['date'].dt.isocalendar().week
train['day'] = train['date'].dt.day
train['quarter'] = train['date'].dt.quarter
train['day_of_week'] = train['date'].dt.dayofweek
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

**2. Feature Engineering**

In [24]:
train

,date,store,item,sales,year,month,week,day,quarter,day_of_week,is_weekend
0,2013-01-01,1,1,13,2013,1,1,1,1,1,0
1,2013-01-02,1,1,11,2013,1,1,2,1,2,0
2,2013-01-03,1,1,14,2013,1,1,3,1,3,0
3,2013-01-04,1,1,13,2013,1,1,4,1,4,0
4,2013-01-05,1,1,10,2013,1,1,5,1,5,1
...,...,...,...,...,...,...,...,...,...,...,...
116001,2015-08-22,4,7,66,2015,8,34,22,3,5,1
116002,2015-08-23,4,7,79,2015,8,34,23,3,6,1
116003,2015-08-24,4,7,61,2015,8,35,24,3,0,0
116004,2015-08-25,4,7,64,2015,8,35,25,3,1,0


In [25]:
train = train.sort_values(['store', 'item', 'date']).reset_index(drop=True)

In [26]:
group = train.groupby(['store','item'])['sales']
train['lag_1'] = group.shift(1)
train['lag_7'] = group.shift(7)
train['lag_14'] = group.shift(14)
train['lag_28'] = group.shift(28)

In [27]:

train['rolling_mean_7'] = (train.groupby(['store','item'])['sales'].
                           shift(1).rolling(7).mean())
train['rolling_mean_28'] = (train.groupby(['store','item'])['sales'].
                            shift(1).rolling(28).mean())

In [28]:
daily_sales = daily_sales.sort_values('date')

daily_sales['rolling_7'] = (
    daily_sales['sales'].rolling(7).mean()
)

daily_sales['rolling_30'] = (
    daily_sales['sales'].rolling(30).mean())

In [29]:
train = train.dropna().reset_index(drop=True)

In [30]:
train_data = train[train['date'] < '2017-01-01']
valid_data = train[train['date'] >= '2017-01-01']

In [31]:
valid_data.sample(4)

,date,store,item,sales,year,month,week,day,quarter,day_of_week,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28
5159,2017-05-11,1,3,34,2017,5,19,11,2,3,0,44.0,43.0,41.0,39.0,45.000000,42.285714
96148,2017-10-07,9,2,89,2017,10,40,7,4,5,1,80.0,93.0,93.0,76.0,70.714286,74.500000
113962,2017-04-24,10,6,65,2017,4,17,24,2,0,0,91.0,66.0,67.0,55.0,81.714286,78.464286
92435,2017-06-12,8,6,78,2017,6,24,12,2,0,0,117.0,79.0,75.0,70.0,99.714286,96.928571


**2. Baseline**

In [32]:
valid_data['pred_baseline'] = valid_data['lag_7']

/tmp/ipykernel_1236/2719664835.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_data['pred_baseline'] = valid_data['lag_7']


In [33]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
mse = mean_squared_error(valid_data['sales'],valid_data['pred_baseline']) ** 0.5
mae = mean_absolute_error(valid_data['sales'],valid_data['pred_baseline'])
print(f"MSE: {mse}")
print(f"MAE: {mae}")

MSE: 9.668904525363327
MAE: 7.3090671885192435


In [34]:
features = ['store','item','year','month','week','day','day_of_week','quarter','is_weekend',
            'lag_1','lag_7','lag_14','lag_28','rolling_mean_7','rolling_mean_28'
          ]

In [35]:
x_train = train_data[features]
x_valid = valid_data[features]
y_train = train_data['sales']
y_valid = valid_data['sales']

In [37]:
categorical_features = ['store', 'item']

numerical_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'year',
    'month',
    'week',
    'day',
    'quarter',
    'day_of_week',
    'is_weekend'
]

In [38]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

In [39]:
X_train_cat = encoder.fit_transform(
    train_data[categorical_features]
)

In [40]:
X_valid_cat = encoder.transform(
    valid_data[categorical_features]
)

In [41]:

X_train_num = train_data[numerical_features].values
X_valid_num = valid_data[numerical_features].values

X_train_final = np.hstack([
    X_train_num,
    X_train_cat
])

X_valid_final = np.hstack([
    X_valid_num,
    X_valid_cat
])

**Model Traning**

In [42]:
from xgboost import XGBRegressor

xgb_ohe = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

xgb_ohe.fit(
    X_train_final,
    y_train
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [43]:
y_valid_pred = xgb_ohe.predict(X_valid_final)

In [44]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(y_valid, y_valid_pred)

rmse = np.sqrt(
    mean_squared_error(y_valid, y_valid_pred)
)

r2 = r2_score(y_valid, y_valid_pred)

mask = y_valid != 0

mape = np.mean(
    np.abs(
        (y_valid[mask] - y_valid_pred[mask])
        / y_valid[mask]
    )
)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)
print("MAPE:", mape)

MAE : 5.160703659057617
RMSE: 6.772307938633168
R²  : 0.9264035224914551
MAPE: 0.15288719931180875


**5. XGBOOST**

In [45]:
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=500,learning_rate=0.05,max_depth=7,random_state=42)
xgb.fit(x_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [46]:
xgb_pred = xgb.predict(x_valid)
xgb_mse = mean_squared_error(y_valid,xgb_pred) ** 0.5
xgb_mae = mean_absolute_error(y_valid,xgb_pred)
print(f"MSE: {xgb_mse}")
print(f"MAE: {xgb_mae}")

MSE: 6.7930021630683415
MAE: 5.177284240722656


**6. LigthXBoosting**

In [47]:
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(n_estimators=500,learning_rate=0.05,random_state=42)
lgb.fit(x_train,y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009422 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1179
[LightGBM] [Info] Number of data points in the train set: 91219, number of used features: 15
[LightGBM] [Info] Start training from score 36.740854


LGBMRegressor(learning_rate=0.05, n_estimators=500, random_state=42)

In [48]:
lgb_pred = lgb.predict(x_valid)
lgb_mse = mean_squared_error(y_valid,lgb_pred) ** 0.5
lgb_mae = mean_absolute_error(y_valid,lgb_pred)
print(f"MSE: {lgb_mse}")
print(f"MAE: {lgb_mae}")

MSE: 6.739109524227914
MAE: 5.141939772872511


In [49]:
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score,mean_absolute_percentage_error

models = {
    'XGBoost': xgb,
    'LightGBM': lgb
}

for name, model in models.items():

    # Training prediction
    train_pred = model.predict(x_train)

    # Validation prediction
    valid_pred = model.predict(x_valid)

    # Training metrics
    train_mae = mean_absolute_error(y_train, train_pred)
    train_rmse = mean_squared_error(y_train, train_pred) ** 0.5
    train_r2_sc = r2_score(y_train,train_pred)
    train_mape = mean_absolute_percentage_error(y_train,train_pred) * 100

    # Validation metrics
    valid_mae = mean_absolute_error(y_valid, valid_pred)
    valid_rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
    valid_r2_sc = r2_score(y_valid,valid_pred)
    valid_mape = mean_absolute_percentage_error(y_valid,valid_pred) * 100

    # Print metrics

    print(f"\n{name}")
    print(f"Train MAE : {train_mae:.4f}")
    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Train R2: {train_r2_sc:.4f}")
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Valid MAE : {valid_mae:.4f}")
    print(f"Valid RMSE: {valid_rmse:.4f}")
    print(f"Valid R2: {valid_r2_sc:.4f}")
    print(f"Valid MAPE: {valid_mape:.2f}")


XGBoost
Train MAE : 4.2051
Train RMSE: 5.4232
Train R2: 0.9398
Train MAPE: 50359199334400.0000
Valid MAE : 5.1773
Valid RMSE: 6.7930
Valid R2: 0.9260
Valid MAPE: 15.35

LightGBM
Train MAE : 4.5389
Train RMSE: 5.8651
Train R2: 0.9296
Train MAPE: 49634858577819.0156
Valid MAE : 5.1419
Valid RMSE: 6.7391
Valid R2: 0.9271
Valid MAPE: 15.32


**Hyper Parameter Tunning**

In [50]:
from sklearn.model_selection import TimeSeriesSplit,RandomizedSearchCV
from xgboost import XGBRegressor
tscv = TimeSeriesSplit(n_splits=3)

In [51]:
xgb = XGBRegressor(objective = 'reg:squarederror',random_state=42,n_jobs=2)

In [52]:
params_grid = {
    'n_estimators': [300, 500, 700],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}


In [53]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=params_grid,
    n_iter=15,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=2,
    random_state=42,
    n_jobs=1
)

In [54]:
random_search.fit(x_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=7, min_child_weight=1, n_estimators=500, subsample=0.8; total time=   1.8s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=7, min_child_weight=1, n_estimators=500, subsample=0.8; total time=   2.7s
[CV] END colsample_bytree=0.8, learning_rate=0.03, max_depth=7, min_child_weight=1, n_estimators=500, subsample=0.8; total time=   5.4s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=3, min_child_weight=5, n_estimators=500, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=3, min_child_weight=5, n_estimators=500, subsample=0.8; total time=   1.4s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=3, min_child_weight=5, n_estimators=500, subsample=0.8; total time=   2.1s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=9, min_child_weight=3, n_estimators=700, subsample=0.8; total tim

RandomizedSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=None),
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=True,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma...
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=2,
                                          num_parallel_tree=None, ...),
                   n_iter=15, n_jobs=1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1.0],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.1],
                                        'max_depth': [3, 5, 7, 9],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [300, 500, 700],
                                        'subsample': [0.7, 0.8, 1.0]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [55]:
best_params = random_search.best_params_
print("Best Hyperparameters:", best_params)

Best Hyperparameters: {'subsample': 0.7, 'n_estimators': 700, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}


In [56]:
valid_pred = random_search.predict(x_valid)

In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

valid_mae = mean_absolute_error(y_valid, valid_pred)

valid_rmse = mean_squared_error(
    y_valid,
    valid_pred
) ** 0.5

print("Tuned XGBoost")
print("Validation MAE :", valid_mae)
print("Validation RMSE:", valid_rmse)

Tuned XGBoost
Validation MAE : 5.141116142272949
Validation RMSE: 6.734117826078469


There is no more difference in xg boost before hyper tunning and after hyper tunning in mae and msq.

**Final Test**

In [58]:

test = pd.read_csv('test.csv')

train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])

test = test.sort_values(['date', 'store', 'item']).reset_index(drop=True)

history = train[['date', 'store', 'item', 'sales']].copy()

history = history.sort_values(
    ['store', 'item', 'date']
).reset_index(drop=True)

In [59]:
print(test.shape)
print(test['date'].nunique())

(45000, 4)
90


In [60]:
predictions = []

xgb.fit(x_train, y_train) # Fit the model before making predictions

for current_date in sorted(test['date'].unique()):

    print("Predicting:", current_date)

    # Today's 500 test rows
    today = test[test['date'] == current_date].copy()

    # --------------------------------
    # Create lag features
    # --------------------------------

    for lag in [1, 7, 14, 28]:

        lookup = history[['date', 'store', 'item', 'sales']].copy()

        # Move historical date forward
        # so it matches today's date

        lookup['date'] = ( lookup['date'] + pd.Timedelta(days=lag))

        lookup = lookup.rename(
            columns={
                'sales': f'lag_{lag}'
            }
        )

        today = today.merge(
            lookup[
                ['date', 'store', 'item', f'lag_{lag}']
            ],
            on=['date', 'store', 'item'],
            how='left'
        )

    # --------------------------------
    # Rolling mean 7
    # --------------------------------

    rolling_7 = (
        history
        .groupby(['store', 'item'])['sales']
        .rolling(7)
        .mean()
        .groupby(level=[0, 1])
        .last()
        .reset_index()
    )

    rolling_7 = rolling_7.rename(
        columns={'sales': 'rolling_mean_7'}
    )

    today = today.merge(
        rolling_7,
        on=['store', 'item'],
        how='left'
    )

    # --------------------------------
    # Rolling mean 28
    # --------------------------------

    rolling_28 = (
        history
        .groupby(['store', 'item'])['sales']
        .rolling(28)
        .mean()
        .groupby(level=[0, 1])
        .last()
        .reset_index()
    )

    rolling_28 = rolling_28.rename(
        columns={'sales': 'rolling_mean_28'}
    )

    today = today.merge(
        rolling_28,
        on=['store', 'item'],
        how='left'
    )

    # --------------------------------
    # Date features
    # --------------------------------

    today['year'] = today['date'].dt.year
    today['month'] = today['date'].dt.month
    today['week'] = today['date'].dt.isocalendar().week.astype(int)
    today['day'] = today['date'].dt.day
    today['quarter'] = today['date'].dt.quarter
    today['day_of_week'] = today['date'].dt.dayofweek

    today['is_weekend'] = (
        today['day_of_week'].isin([5, 6])
    ).astype(int)

    # --------------------------------
    # Select XGBoost features
    # --------------------------------

    X_today = today[features]
    print(X_today)
    # --------------------------------
    # Predict today's sales
    # --------------------------------

    today['sales'] = xgb.predict(X_today)

    # --------------------------------
    # Save today's predictions
    # --------------------------------

    predictions.append(today)

    # --------------------------------
    # Add predictions to history
    # --------------------------------

    history = pd.concat(
        [
            history,
            today[
                ['date', 'store', 'item', 'sales']
            ]
        ],
        ignore_index=True
    )

Predicting: 2018-01-01 00:00:00
     store  item  year  month  week  day  day_of_week  quarter  is_weekend  \
0        1     1  2018      1     1    1            0        1           0   
1        1     2  2018      1     1    1            0        1           0   
2        1     3  2018      1     1    1            0        1           0   
3        1     4  2018      1     1    1            0        1           0   
4        1     5  2018      1     1    1            0        1           0   
..     ...   ...   ...    ...   ...  ...          ...      ...         ...   
495     10    46  2018      1     1    1            0        1           0   
496     10    47  2018      1     1    1            0        1           0   
497     10    48  2018      1     1    1            0        1           0   
498     10    49  2018      1     1    1            0        1           0   
499     10    50  2018      1     1    1            0        1           0   

     lag_1  lag_7  lag_14  lag_

In [61]:
predictions_df = pd.concat(predictions,ignore_index=True)

In [62]:
print(predictions_df.head())
print(predictions_df.shape)
print(predictions_df['sales'].describe())
print(predictions_df.isna().sum())

     id       date  store  item  lag_1  lag_7  lag_14  lag_28  rolling_mean_7  \
0     0 2018-01-01      1     1   23.0   13.0    19.0     7.0       18.142857   
1   900 2018-01-01      1     2   67.0   37.0    37.0    33.0       51.142857   
2  1800 2018-01-01      1     3   29.0   26.0    21.0    21.0       26.428571   
3  2700 2018-01-01      1     4   15.0   11.0    11.0    10.0       17.285714   
4  3600 2018-01-01      1     5   17.0   12.0    10.0    13.0       14.142857   

   rolling_mean_28  year  month  week  day  quarter  day_of_week  is_weekend  \
0        16.678571  2018      1     1    1        1            0           0   
1        46.285714  2018      1     1    1        1            0           0   
2        27.785714  2018      1     1    1        1            0           0   
3        15.821429  2018      1     1    1        1            0           0   
4        13.785714  2018      1     1    1        1            0           0   

       sales  
0  12.291924  
1 

In [63]:
predictions_df.head()

,id,date,store,item,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,year,month,week,day,quarter,day_of_week,is_weekend,sales
0,0,2018-01-01,1,1,23.0,13.0,19.0,7.0,18.142857,16.678571,2018,1,1,1,1,0,0,12.291924
1,900,2018-01-01,1,2,67.0,37.0,37.0,33.0,51.142857,46.285714,2018,1,1,1,1,0,0,35.323124
2,1800,2018-01-01,1,3,29.0,26.0,21.0,21.0,26.428571,27.785714,2018,1,1,1,1,0,0,21.852089
3,2700,2018-01-01,1,4,15.0,11.0,11.0,10.0,17.285714,15.821429,2018,1,1,1,1,0,0,12.676690
4,3600,2018-01-01,1,5,17.0,12.0,10.0,13.0,14.142857,13.785714,2018,1,1,1,1,0,0,10.655706
